In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os
import sys
sys.path.insert(0, os.path.abspath('../'))
import ResParTools as pt

# 2. Построить топологию acpype

Вход — SDF из ноутбука 1 со свойствами RDKit и водородами, добавленными для топологии. acpype
задаёт атомные типы и недостающие bonded-параметры; его частичные заряды не подменяют выбранные
Espaloma/RESP заряды.

In [ ]:
# Параметры модификации (заполняются генератором)
FOLDER = 'Lysine_prop'
PTM_name = 'Lysine_prop'
base_name = 'KPR'
monomer_file = 'molecules/K_prop.smiles'
OUT = os.path.abspath(os.path.join('..', 'build', 'notebooks', FOLDER))
INPUT_BASE = os.path.join(OUT, f'4_{os.path.splitext(os.path.basename(monomer_file))[0]}.sdf')
INPUT_H = os.path.join(OUT, f'4_1_{os.path.splitext(os.path.basename(monomer_file))[0]}.sdf')
ACPYPE_DIR = os.path.join(OUT, 'acpype')
os.makedirs(ACPYPE_DIR, exist_ok=True)

## Шаг 1. Загрузить SDF и проверить входные атомы

In [ ]:
residue = pt.load_mol(INPUT_BASE)  # без HW: эта копия пойдёт в RTP
acpype_residue = pt.load_mol(INPUT_H)  # HW нужны для корректной валентности acpype
assert residue.GetProp('ChargeMethod') in ('Espaloma', 'RESP')
names = [a.GetProp('AtomName') for a in residue.GetAtoms()]
assert len(names) == len(set(names)), 'Имена AtomName должны быть уникальны.'
assert acpype_residue.GetNumAtoms() > residue.GetNumAtoms()
assert [a.GetProp('AtomName') for a in list(acpype_residue.GetAtoms())[:residue.GetNumAtoms()]] == names
print('Остаток:', residue.GetProp('ResName'), 'атомов:', residue.GetNumAtoms(),
      'заряды:', residue.GetProp('ChargeMethod'),
      '| атомов с HW для acpype:', acpype_residue.GetNumAtoms())

## Шаг 2. Запустить acpype

`run_acpype` сверяет число и элементы атомов ITP с входной молекулой и записывает `AcpypeType` и
путь `AcpypeItp` в свойства.

In [ ]:
acpype_residue = pt.run_acpype(acpype_residue, workdir=ACPYPE_DIR)
for atom, atom_h in zip(residue.GetAtoms(), acpype_residue.GetAtoms()):
    if atom.GetProp('AtomName') != atom_h.GetProp('AtomName') or atom.GetSymbol() != atom_h.GetSymbol():
        raise ValueError(f'Порядок атомов SDF и acpype-версии различается на {atom.GetIdx()}')
    atom.SetProp('AcpypeType', atom_h.GetProp('AcpypeType'))
residue.SetProp('AcpypeItp', acpype_residue.GetProp('AcpypeItp'))
if acpype_residue.HasProp('AcpypeCapMap'):
    residue.SetProp('AcpypeCapMap', acpype_residue.GetProp('AcpypeCapMap'))
itp = residue.GetProp('AcpypeItp')
print('ITP:', itp)
atom_table = [(a.GetIdx(), a.GetProp('AtomName'), a.GetSymbol(), a.GetProp('AcpypeType'))
              for a in residue.GetAtoms()]
for row in atom_table:
    print(*row)

## Сохранить топологические типы

SDF сохраняет имена и заряды предыдущего шага вместе с типами acpype. Файл ITP остаётся в папке
`build/notebooks/<модификация>/acpype/`.

In [ ]:
OUTPUT = pt.save_mol(residue, os.path.join(OUT, f'5_{os.path.splitext(os.path.basename(monomer_file))[0]}'))
pt.save_mol(acpype_residue, os.path.join(OUT, f'5_1_{os.path.splitext(os.path.basename(monomer_file))[0]}'))
print('SDF остатка без HW для RTP:', OUTPUT)
print('SDF с HW только для acpype:', os.path.join(OUT, f'5_1_{os.path.splitext(os.path.basename(monomer_file))[0]}.sdf'))